In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import re

In [2]:
file_path = "348950232_按文本_问卷_184_179.xlsx"
print("正在加载数据...")
df_raw = pd.read_excel(file_path)

# 2.1 陷阱题清洗 (Q5)
# 逻辑：Q5 题干中明确要求选“方案B”。如果没选B，视为无效样本。
trap_col = [c for c in df_raw.columns if 'Q5' in c or '显示测试' in c][0]
valid_df = df_raw[df_raw[trap_col].str.contains('方案B', na=False)].copy()

print(f"原始样本量: {len(df_raw)}")
print(f"清洗后有效样本量: {len(valid_df)} (剔除率: {1 - len(valid_df)/len(df_raw):.1%})")

正在加载数据...
原始样本量: 179
清洗后有效样本量: 152 (剔除率: 15.1%)


In [3]:
# ==========================================
# 3. 属性解析函数
# ==========================================
def parse_attributes(text, is_none=False):
    """
    将问卷中的文本描述解析为数值型水平。
    """
    if is_none:
        # 关键逻辑：都不选时，所有功能属性归0，价格归0
        return {'Smart': 0, 'Context': 0, 'Privacy': 0, 'Price': 0, 'ASC': 0}
    
    # 解析智能水平 (Base=1)
    if 'LV3' in text or '专家' in text: smart = 3
    elif 'LV2' in text or '进阶' in text: smart = 2
    else: smart = 1  # 基础水平
    
    # 解析上下文 (Base=1)
    if 'LV3' in text or '全资料' in text: context = 3
    elif 'LV2' in text or '整本' in text: context = 2
    else: context = 1 # 标准
    
    # 解析隐私 (Base=1)
    if 'LV2' in text or '严格保密' in text: privacy = 2
    else: privacy = 1 # 默认开启
    
    # 解析价格
    # 使用正则表达式提取数字
    price_match = re.search(r'(\d+)元', text)
    price = int(price_match.group(1)) if price_match else 0
    
    return {'Smart': smart, 'Context': context, 'Privacy': privacy, 'Price': price, 'ASC': 1}

# ==========================================
# 4. 数据转换 (Wide to Long)
# ==========================================
print("正在重构数据结构...")
long_data = []
choice_cols = [c for c in df_raw.columns if '方案A' in c and 'Q5' not in c]

for idx, row in valid_df.iterrows():
    respondent_id = row['序号'] # 或其他ID列
    
    for q_col in choice_cols:
        # 提取题干中的属性描述
        # 假设题干格式固定："....方案A：【智能】...方案B：..."
        try:
            parts = q_col.split('方案B：')
            text_a = parts[0].split('方案A：')[1]
            text_b = parts[1]
        except IndexError:
            continue # 跳过格式不对的列
            
        user_choice = str(row[q_col])
        
        # 构建三个选项：A, B, None
        options = [
            ('方案A', parse_attributes(text_a, is_none=False)),
            ('方案B', parse_attributes(text_b, is_none=False)),
            ('都不选', parse_attributes("", is_none=True))
        ]
        
        for alt_label, attrs in options:
            # 判断是否被选中
            choice = 1 if alt_label in user_choice else 0
            
            # 存入列表
            long_data.append({
                'ID': respondent_id,
                'Question': q_col[:10], # 简略题号
                'Alt_Label': alt_label,
                'Choice': choice,
                **attrs
            })

df_long = pd.DataFrame(long_data)

def remove_collinear_features(X):
    """
    自动检测并剔除导致奇异矩阵（Singular Matrix）的共线性列。
    基于矩阵秩 (Matrix Rank) 进行判断。
    """
    import numpy as np
    
    # 必须保留的列（如果它们存在），建议放在最前面检查
    # 这里我们不强制重排，但依赖输入顺序。建议 ASC 和 Price 放前面。
    
    kept_cols = []
    dropped_cols = []
    
    # 遍历每一列，检查加入后是否增加了矩阵的“秩” (Rank)
    for col in X.columns:
        # 尝试将当前列加入已选列表
        current_cols = kept_cols + [col]
        
        # 计算当前子矩阵的秩
        # matrix_rank 需要数值型 numpy 数组
        rank = np.linalg.matrix_rank(X[current_cols].values)
        
        # 如果秩等于列数，说明该列独立，有新信息 -> 保留
        if rank == len(current_cols):
            kept_cols.append(col)
        # 如果秩小于列数，说明该列是前面列的线性组合 -> 剔除
        else:
            dropped_cols.append(col)
            
    if dropped_cols:
        print(f"⚠️ 自动剔除了 {len(dropped_cols)} 个共线性变量以修复模型: {dropped_cols}")
        print("   (原因: 这些变量在当前样本中全是0，或是其他变量的线性组合)")
    else:
        print("✅ 数据矩阵健康，未发现完全共线性。")
        
    return X[kept_cols]

# ==========================================
# 5. 虚拟变量编码 (核心计量逻辑)
# ==========================================
# 必须手动处理 Dummy 变量，以配合 ASC 模型
# 逻辑：
# Smart_2 = 1 当且仅当 Smart == 2 (且不是None)
# Smart_3 = 1 当且仅当 Smart == 3
# Smart_1 不需要进入模型，因为它被 ASC 吸收了 (作为基准产品的效用)
# Smart_0 (None) 不需要进入模型，因为它对应 ASC=0

df_model = df_long.copy()

# 创建特定水平的虚拟变量
df_model['Smart_2'] = (df_model['Smart'] == 2).astype(int)
df_model['Smart_3'] = (df_model['Smart'] == 3).astype(int)

df_model['Context_2'] = (df_model['Context'] == 2).astype(int)
df_model['Context_3'] = (df_model['Context'] == 3).astype(int)

df_model['Privacy_2'] = (df_model['Privacy'] == 2).astype(int)

# 定义自变量 (X) 和 因变量 (y)
# 注意：不包含常数项 (const)，因为 ASC 充当了 Buy 选项的截距
X_cols = ['ASC', 'Price', 'Smart_2', 'Smart_3', 'Context_2', 'Context_3', 'Privacy_2']
X = df_model[X_cols]
X = remove_collinear_features(X)
y = df_model['Choice']

# ==========================================
# 6. 模型拟合 (Statsmodels)
# ==========================================
print("正在拟合 Logit 模型...")
try:
    logit_model = sm.Logit(y, X)
    result = logit_model.fit(disp=0) # disp=0 不打印迭代过程
    
    print("\n" + "="*30)
    print("      回归分析结果摘要")
    print("="*30)
    print(result.summary())
    
    # ==========================================
    # 7. 支付意愿 (WTP) 自动计算
    # ==========================================
    print("\n" + "="*30)
    print("      支付意愿 (WTP) 测算")
    print("="*30)
    
    beta_price = result.params['Price']
    
    # 检查价格系数是否正常 (理论上应为负)
    if beta_price >= 0:
        print("警告：价格系数为正，WTP 计算可能无意义。请检查样本量或数据质量。")
    
    def calc_wtp(beta_name, label):
        if beta_name in result.params:
            wtp = -result.params[beta_name] / beta_price
            print(f"【{label}】愿意多付: {wtp:.2f} 元/月")
        else:
            print(f"未找到 {beta_name} 系数")

    calc_wtp('Smart_2', '智能: 进阶推理 (Level 2)')
    calc_wtp('Smart_3', '智能: 专家创作 (Level 3)')
    calc_wtp('Context_2', '上下文: 书籍级 (Level 2)')
    calc_wtp('Context_3', '上下文: 资料库级 (Level 3)')
    calc_wtp('Privacy_2', '隐私: 严格保密')
    
    # 计算 ASC 的货币价值 (进入市场的意愿)
    wtp_base = -result.params['ASC'] / beta_price
    print(f"【基础门槛】用户为了使用基础版AI愿意支付: {wtp_base:.2f} 元/月")

except Exception as e:
    print(f"模型运行出错: {e}")
    print("建议检查：是否存在完全共线性的列？(例如所有样本都选了同一个选项)")

正在重构数据结构...
⚠️ 自动剔除了 3 个共线性变量以修复模型: ['Smart_3', 'Context_2', 'Context_3']
   (原因: 这些变量在当前样本中全是0，或是其他变量的线性组合)
正在拟合 Logit 模型...

      回归分析结果摘要
                           Logit Regression Results                           
Dep. Variable:                 Choice   No. Observations:                 4104
Model:                          Logit   Df Residuals:                     4100
Method:                           MLE   Df Model:                            3
Date:                Wed, 11 Feb 2026   Pseudo R-squ.:                -0.06142
Time:                        14:41:19   Log-Likelihood:                -2772.7
converged:                       True   LL-Null:                       -2612.3
Covariance Type:            nonrobust   LLR p-value:                     1.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ASC           -0.5485      0.123     -4.463      0.000      -0.789  